# 21.2 Patch 如何交換資訊

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/21-attention/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""由15.1的QK-softmax-V走到ViT多頭attention，顯示可核對的權重列。"""

import json
import math

import torch
from torch.nn import functional as F

from miniyolo.vision_data import ColorRectangles
from miniyolo.vision_transformer import SelfAttention, TinyViT


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    tokens = torch.tensor([[[1., 0.], [0., 1.], [1., 1.], [0., 0.]]])
    attention = SelfAttention(embed_dim=2, num_heads=1)
    with torch.no_grad():
        attention.qkv.weight.copy_(torch.eye(2).repeat(3, 1))
        attention.qkv.bias.zero_()
        attention.projection.weight.copy_(torch.eye(2))
        attention.projection.bias.zero_()
    output, weights = attention(tokens, return_attention=True)
    expected_row = (torch.tensor([1., 0., 1., 0.]) / math.sqrt(2)).softmax(0)
    assert torch.allclose(weights[0, 0, 0], expected_row)
    assert torch.allclose(output[0, 0], torch.tensor([expected_row[0] + expected_row[2], expected_row[1] + expected_row[2]]))
    assert torch.allclose(weights.sum(-1), torch.ones(1, 1, 4))
    before = attention.qkv.weight.detach().clone()
    optimizer = torch.optim.SGD(attention.parameters(), lr=.1)
    loss = F.mse_loss(output, tokens * torch.tensor([1., .5]))
    optimizer.zero_grad()
    loss.backward()
    gradient_sums = [float(part.abs().sum()) for part in attention.qkv.weight.grad.chunk(3, dim=0)]
    assert all(value > 0 for value in gradient_sums)
    assert all(parameter.grad is not None and torch.isfinite(parameter.grad).all() for parameter in attention.parameters())
    optimizer.step()
    assert not torch.equal(before, attention.qkv.weight)
    model = TinyViT().eval()
    image = ColorRectangles(samples=128).images[3:4]
    embedded = model.embed_tokens(image)
    # 第一個block的attention先看LayerNorm後的tokens；沒有預訓練。
    mixed, multi_weights = model.blocks[0].attention(model.blocks[0].norm1(embedded), return_attention=True)
    assert multi_weights.shape == (1, 4, 17, 17)
    assert torch.allclose(multi_weights.sum(-1), torch.ones(1, 4, 17), atol=1e-6)
    with torch.no_grad():
        # 交換第1與第16個patch，CLS仍在第0位；關閉dropout後檢查排列關係。
        permutation = torch.arange(17)
        permutation[1], permutation[16] = 16, 1
        contents = model.embed_tokens(image, use_position=False)
        without_pos = model.encode_tokens(contents)
        swapped_without_pos = model.encode_tokens(contents[:, permutation])
        cls_without_pos_change = float((without_pos[:, 0] - swapped_without_pos[:, 0]).abs().max())
        patches_equivariance_error = float((swapped_without_pos[:, 1:] - without_pos[:, permutation][:, 1:]).abs().max())
        assert torch.allclose(swapped_without_pos, without_pos[:, permutation], atol=2e-6)
        # 位置保持原槽，只換patch內容。
        fixed_pos_change = float((model.encode_tokens(contents + model.pos_embed)[:, 0]
                                  - model.encode_tokens(contents[:, permutation] + model.pos_embed)[:, 0]).abs().max())
        assert fixed_pos_change > 1e-6
    result = {
        "event": "attention", "manual_tokens": tokens[0].tolist(),
        "first_query_scores_before_scale": [1., 0., 1., 0.], "scale": math.sqrt(2),
        "first_attention_row": weights[0, 0, 0].detach().tolist(),
        "first_weighted_value": output[0, 0].detach().tolist(),
        "manual_weight_shape": list(weights.shape), "manual_loss": float(loss.detach()),
        "q_k_v_gradient_abs_sums": gradient_sums, "qkv_weights_changed": True,
        "vit_tokens_shape": list(embedded.shape), "vit_heads": 4, "vit_head_dim": 8,
        "vit_attention_shape": list(multi_weights.shape), "vit_attention_scores": multi_weights.numel(),
        "vit_output_shape": list(mixed.shape),
        "vit_row_sum_max_error": float((multi_weights.sum(-1) - 1).abs().max().detach()),
        "swap_patch_sequence_indices": [1, 16],
        "without_position_cls_max_change": cls_without_pos_change,
        "without_position_patch_equivariance_max_error": patches_equivariance_error,
        "with_fixed_position_cls_max_change": fixed_pos_change,
        "limitation": "One SGD update validates the mechanism; random ViT attention weights are not explanations of a trained decision.",
    }
    print(json.dumps(result, ensure_ascii=False, allow_nan=False))
    return result


if __name__ == "__main__":
    main()


{"event": "attention", "manual_tokens": [[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [0.0, 0.0]], "first_query_scores_before_scale": [1.0, 0.0, 1.0, 0.0], "scale": 1.4142135623730951, "first_attention_row": [0.3348807692527771, 0.1651192307472229, 0.3348807692527771, 0.1651192307472229], "first_weighted_value": [0.6697615385055542, 0.5], "manual_weight_shape": [1, 1, 4, 4], "manual_loss": 0.1594690978527069, "q_k_v_gradient_abs_sums": [0.07701923698186874, 0.07701923698186874, 0.4417698085308075], "qkv_weights_changed": true, "vit_tokens_shape": [1, 17, 32], "vit_heads": 4, "vit_head_dim": 8, "vit_attention_shape": [1, 4, 17, 17], "vit_attention_scores": 1156, "vit_output_shape": [1, 17, 32], "vit_row_sum_max_error": 1.7881393432617188e-07, "swap_patch_sequence_indices": [1, 16], "without_position_cls_max_change": 2.384185791015625e-07, "without_position_patch_equivariance_max_error": 2.384185791015625e-07, "with_fixed_position_cls_max_change": 0.0016033649444580078, "limitation": "One SGD upd